# Fabric IQ solution accelerator
This notebook orchestrates the end-to-end deployment of **[TBD](https://github.com/microsoft/tbd/)** into the current Microsoft Fabric workspace using the [`fabric-launcher`](https://github.com/microsoft/fabric-launcher) library.

**Installation tasks**
This notebook performs the following tasks:
1. **🚀 Deployment**: download source code, deploy Fabric items, load reference Data
1. **✅ Post-Deployment Tasks**: Complete post-deployment configuration tasks, such as:
    - Import sample data

**Expected duration**
This notebook will take about 12 minutes to run

## 🚀 Deployment
Download source code, deploy Fabric items, load sample data

In [ ]:
%pip install fabric-launcher --quiet
import notebookutils
notebookutils.session.restartPython()

In [ ]:
from fabric_launcher import FabricLauncher

launcher = FabricLauncher(notebookutils,
    api_root_url = "https://api.fabric.microsoft.com" #Default is https://api.fabric.microsoft.com, but may vary depending on your environment
    )


# ── GitHub source settings ────────────────────────────────────────────────────
GITHUB_OWNER     = "microsoft"
GITHUB_REPO      = "microsoft-iq-solution-accelerator"
GITHUB_BRANCH    = "main"
GITHUB_FABRIC_WORKSPACE_PATH = "src/fabric/fabric_workspace"  # Folder inside the repo where the fabric solution is synched

# ── Lakehouse data settings ─────────────────────────────────────────
LAKEHOUSE_NAME = "miqsadata"                # Lakehouse in the workspace where the ingested CSV files will be landed
GITHUB_FABRIC_INFRA_PATH = "infra/fabric"   # Folder inside the repo containing the data to be ingested in the Lakehouse
DATA_FOLDERS={
    f"{GITHUB_FABRIC_INFRA_PATH}/data": "/data" # Key: folder inside the GitHub repo where the source CSV files are located; Value: target folder in the Lakehouse where the CSV files will be landed
    }

# ── Ontology deployment settings ────────────────────────────────────────────
ONTOLOGY_NAMES = [  # List of ontology names to deploy from the repository
    "RetailSupplyChainOntologyModel"
    ]
ONTOLOGY_TARGET_FOLDER = "ontology" # Target folder where ontologies will be moved after deployment

# ── Source environment GUIDs (from parameter.yml) ───────────────────────────
# These are the hardcoded GUIDs from the source DEV workspace that need to be replaced
# during ontology deployment with actual target workspace/lakehouse IDs
SOURCE_WORKSPACE_ID = "27818fad-0450-42b9-a4a0-db84075ac8d7"
SOURCE_LAKEHOUSE_ID = "e6cd0b76-50e8-498f-9cd0-89251fae70c2"

In [ ]:
# Deploy solution with data folders
launcher.download_and_deploy(
    repo_owner=GITHUB_OWNER,
    repo_name=GITHUB_REPO,
    branch = GITHUB_BRANCH,
    workspace_folder=GITHUB_FABRIC_WORKSPACE_PATH,
    allow_non_empty_workspace=True,
    item_type_stages = [
        ["Lakehouse"],
        ["Notebook", "SemanticModel", "Report"]
    ],
    validate_after_deployment=True,
    generate_report=False,
    data_folders=DATA_FOLDERS,
    lakehouse_name=LAKEHOUSE_NAME
)

## ✅ Automated Post-Deployment Tasks

In [ ]:
#Import required libraries
import sempy.fabric as fabric
from fabric_launcher import move_item_to_folder
from fabric_launcher.post_deployment_utils import create_or_update_fabric_item, scan_logical_ids
from pathlib import Path

# Initialize Fabric client and workspace
client = fabric.FabricRestClient()
workspace_id = fabric.get_workspace_id()

# Calculate repository paths once for consistent use
workspace_directory = launcher.repository_path  # Path to workspace items (src/fabric/fabric_workspace)
workspace_folder_parts = Path(GITHUB_FABRIC_WORKSPACE_PATH).parts
repo_root = Path(workspace_directory)
for _ in range(len(workspace_folder_parts)):
    repo_root = repo_root.parent
repo_root_directory = str(repo_root)  # Repository root for ontology deployment

print(f"📁 Workspace directory: {workspace_directory}")
print(f"📁 Repository root: {repo_root_directory}")

In [ ]:
# Run a notebook to: Create tables
result = launcher.run_notebook_synchronous(
    notebook_name="pipeline_main",
    parameters={},
    timeout_seconds=3600
)

In [ ]:
# Deploy Ontology item from repository
import os, json, base64, time
from fabric_launcher.post_deployment_utils import get_sql_endpoint
print("📦 Deploying Ontology item...")

def deploy_ontology(ontology_name,
                    repository_directory,
                    client=client,
                    workspace_id=workspace_id):
    """Deploy an Ontology item via the Fabric REST API with logical ID and URI replacement."""

    # Resolve ontology directory and detect folder structure
    ontology_folder = f"{ontology_name}.Ontology"
    
    # Search for the ontology directory in repository root and all subfolders
    ontology_dir = None
    folder_path = None
    
    for root, dirs, files in os.walk(repository_directory):
        if ontology_folder in dirs:
            ontology_dir = os.path.join(root, ontology_folder)
            # Extract folder path - look for workspace, fabric_workspace, or definitions patterns
            # to determine the target folder structure in Fabric
            rel_path = os.path.relpath(root, repository_directory)
            
            # Check if this is in a recognized workspace structure
            path_parts = rel_path.split(os.sep)
            if 'workspace' in path_parts:
                # Extract folder path relative to workspace directory
                workspace_idx = path_parts.index('workspace')
                if len(path_parts) > workspace_idx + 1:
                    folder_path = '/'.join(path_parts[workspace_idx + 1:])
            elif 'fabric_workspace' in path_parts:
                # Extract folder path relative to fabric_workspace directory
                workspace_idx = path_parts.index('fabric_workspace')
                if len(path_parts) > workspace_idx + 1:
                    folder_path = '/'.join(path_parts[workspace_idx + 1:])
            break
    
    if not ontology_dir:
        raise FileNotFoundError(f"Ontology directory '{ontology_folder}' not found in repository")
    
    print(f"   Ontology directory: {ontology_dir}")
    if folder_path:
        print(f"   Target folder: {folder_path}")

    # Step 1: Scan logical IDs to map logical → real item IDs
    print("1. Scanning logical IDs in repository...")
    logical_id_map = scan_logical_ids(
        repository_directory=repository_directory,
        workspace_id=workspace_id,
        client=client
    )
    print(f"   Found {len(logical_id_map)} logical ID mappings")

    # Step 2: Build lakehouse item ID map (with pagination support)
    print("2. Building lakehouse item ID mapping...")
    lakehouse_id_map = {}
    list_url = f"v1/workspaces/{workspace_id}/lakehouses"
    
    while list_url:
        list_response = client.get(list_url)
        if list_response.status_code == 200:
            response_data = list_response.json()
            lakehouses = response_data.get("value", [])
            
            for lakehouse in lakehouses:
                if lakehouse.get("displayName") == LAKEHOUSE_NAME:
                    lakehouse_id_map[LAKEHOUSE_NAME] = lakehouse.get("id")
                    print(f"   Found lakehouse '{LAKEHOUSE_NAME}': {lakehouse.get('id')}")
                    list_url = None  # Stop pagination once found
                    break
            
            # Get continuation token for next page (if lakehouse not found yet)
            if list_url and not lakehouse_id_map:
                list_url = response_data.get("continuationUri")
        else:
            print(f"   ⚠️ Failed to list lakehouses: HTTP {list_response.status_code}")
            break
    
    if not lakehouse_id_map:
        print(f"   ⚠️ Warning: Could not find lakehouse '{LAKEHOUSE_NAME}' in workspace")

    # Step 3: Resolve lakehouse SQL endpoints for all Lakehouses in the repository
    print("3. Resolving lakehouse SQL endpoints...")
    lakehouse_sql_endpoint_map = _build_lakehouse_sql_endpoint_map(
        repository_directory, workspace_id, logical_id_map, client
    )
    print(f"   Resolved {len(lakehouse_sql_endpoint_map)} lakehouse SQL endpoint(s)")

    # Step 4: Collect all definition parts and perform replacements
    print("4. Collecting ontology definition parts...")
    parts = []

    for root, dirs, files in os.walk(ontology_dir):
        for file_name in files:
            file_path = os.path.join(root, file_name)
            rel_path = os.path.relpath(file_path, ontology_dir).replace("\\", "/")

            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()

            try:
                data = json.loads(content)
            except json.JSONDecodeError:
                payload = base64.b64encode(content.encode("utf-8")).decode("utf-8")
                parts.append({"path": rel_path, "payload": payload, "payloadType": "InlineBase64"})
                continue

            # Serialize to string for bulk replacements
            content_str = json.dumps(data)

            # Replace workspace ID placeholder with actual workspace ID (both formats)
            content_str = content_str.replace("00000000-0000-0000-0000-000000000000", workspace_id)
            # Replace source DEV workspace ID with target workspace ID
            content_str = content_str.replace(SOURCE_WORKSPACE_ID, workspace_id)

            # Replace logical item IDs with real item IDs
            for logical_id, real_id in logical_id_map.items():
                content_str = content_str.replace(logical_id, real_id)

            # Replace source DEV lakehouse GUID with actual deployed lakehouse ID
            if LAKEHOUSE_NAME in lakehouse_id_map:
                actual_lakehouse_id = lakehouse_id_map[LAKEHOUSE_NAME]
                content_str = content_str.replace(SOURCE_LAKEHOUSE_ID, actual_lakehouse_id)

            # Replace SQL endpoint values using the sibling itemId to select the correct lakehouse endpoint
            data_replaced = json.loads(content_str)
            _replace_sql_endpoints(data_replaced, lakehouse_sql_endpoint_map)

            final_content = json.dumps(data_replaced)
            payload = base64.b64encode(final_content.encode("utf-8")).decode("utf-8")
            parts.append({"path": rel_path, "payload": payload, "payloadType": "InlineBase64"})

    print(f"   Collected {len(parts)} definition parts")
    if not parts:
        raise ValueError(f"No definition parts found in {ontology_dir}.")

    # Step 5: Check if ontology already exists in workspace
    print("5. Checking if ontology already exists...")
    existing_ontology_id = None
    response = client.get(f"v1/workspaces/{workspace_id}/items?type=Ontology")
    if response.status_code == 200:
        for item in response.json().get("value", []):
            if item["displayName"] == ontology_name:
                existing_ontology_id = item["id"]
                break

    definition = {"parts": parts}

    # Step 6: Create or update the ontology via the ontology-specific API
    if existing_ontology_id:
        print(f"6. Updating existing ontology '{ontology_name}' (ID: {existing_ontology_id})...")
        response = client.post(
            f"v1/workspaces/{workspace_id}/ontologies/{existing_ontology_id}/updateDefinition",
            json={"definition": definition}
        )
    else:
        print(f"6. Creating new ontology '{ontology_name}'...")
        response = client.post(
            f"v1/workspaces/{workspace_id}/ontologies",
            json={"displayName": ontology_name, "definition": definition}
        )

    # Handle long-running operation (LRO)
    if response.status_code == 202:
        operation_id = response.headers.get("x-ms-operation-id")
        retry_after = int(response.headers.get("Retry-After", 30))
        print(f"   Operation in progress (ID: {operation_id}). Polling...")

        while True:
            time.sleep(retry_after)
            poll_response = client.get(f"v1/operations/{operation_id}")
            if poll_response.status_code == 200:
                status = poll_response.json().get("status", "")
                if status in ("Succeeded", "Completed"):
                    print("   Operation completed successfully.")
                    break
                elif status in ("Failed", "Cancelled"):
                    raise Exception(f"Operation {status}: {poll_response.json()}")
                else:
                    print(f"   Status: {status}, retrying in {retry_after}s...")
            else:
                raise Exception(f"Failed to poll operation: {poll_response.status_code} {poll_response.text}")
    elif response.status_code in (200, 201):
        if not existing_ontology_id:
            existing_ontology_id = response.json().get("id")
        print(f"   Ontology {'updated' if existing_ontology_id else 'created'} successfully.")
    else:
        raise Exception(f"API call failed: {response.status_code} {response.text}")

    # Step 7: Move ontology to the appropriate folder if needed
    if folder_path:
        print(f"7. Moving ontology to folder '{folder_path}'...")
        success = move_item_to_folder(
            item_name=ontology_name,
            item_type="Ontology",
            folder_name=folder_path,
            workspace_id=workspace_id,
            client=client
        )
        if success:
            print(f"   Moved to folder successfully.")
        else:
            print(f"   ⚠️ Failed to move to folder (ontology created but remains at workspace root)")

    print(f"✅ {ontology_name} deployed successfully")
    return existing_ontology_id


def _build_lakehouse_sql_endpoint_map(repository_directory, workspace_id, logical_id_map, client):
    """Scan the repository for Lakehouse items and resolve their SQL endpoints.
    Returns a dict mapping real lakehouse item ID → SQL endpoint connection string."""
    lakehouse_sql_endpoint_map = {}

    for root, dirs, files in os.walk(repository_directory):
        if ".platform" not in files:
            continue
        # Only process Lakehouse folders (e.g. "miqsadata.Lakehouse")
        folder_name = os.path.basename(root)
        if not folder_name.endswith(".Lakehouse"):
            continue

        with open(os.path.join(root, ".platform"), "r", encoding="utf-8") as f:
            platform_data = json.load(f)

        display_name = platform_data.get("metadata", {}).get("displayName")
        logical_id = platform_data.get("config", {}).get("logicalId")
        if not display_name or not logical_id:
            continue

        real_id = logical_id_map.get(logical_id, logical_id)
        try:
            sql_endpoint = get_sql_endpoint(workspace_id, display_name, "Lakehouse", client)
            lakehouse_sql_endpoint_map[real_id] = sql_endpoint
            print(f"   Mapped lakehouse '{display_name}' (ID: {real_id}) → {sql_endpoint}")
        except Exception as e:
            print(f"   ⚠️ Could not resolve SQL endpoint for lakehouse '{display_name}': {e}")

    return lakehouse_sql_endpoint_map


def _replace_sql_endpoints(obj, lakehouse_sql_endpoint_map):
    """Recursively replace SQL endpoint values based on the sibling itemId → lakehouse endpoint mapping.
    
    This function can replace various endpoint-related fields in the ontology definition:
    - sqlEndpoint: SQL endpoint connection string
    - connectionString: Alternative field name for SQL endpoints
    """
    if isinstance(obj, dict):
        # If this dict has both sqlEndpoint/connectionString and itemId, resolve via the mapping
        if "itemId" in obj:
            item_id = obj["itemId"]
            if item_id in lakehouse_sql_endpoint_map:
                # Replace sqlEndpoint if present
                if "sqlEndpoint" in obj:
                    obj["sqlEndpoint"] = lakehouse_sql_endpoint_map[item_id]
                # Replace connectionString if present
                if "connectionString" in obj:
                    obj["connectionString"] = lakehouse_sql_endpoint_map[item_id]
        
        # Recurse into nested dictionaries
        for value in obj.values():
            _replace_sql_endpoints(value, lakehouse_sql_endpoint_map)
    elif isinstance(obj, list):
        for item in obj:
            _replace_sql_endpoints(item, lakehouse_sql_endpoint_map)


# Deploy all ontologies in the array
deployed_ontology_ids = []
for ontology_name in ONTOLOGY_NAMES:
    print(f"\n📦 Deploying ontology '{ontology_name}'...")
    ontology_id = deploy_ontology(
        ontology_name=ontology_name,
        repository_directory=repo_root_directory
    )
    deployed_ontology_ids.append((ontology_name, ontology_id))
    print(f"✅ Ontology '{ontology_name}' deployed successfully (ID: {ontology_id})")

print(f"\n✅ All {len(deployed_ontology_ids)} ontologies deployed successfully")

In [ ]:
# Deploy Data Agent items from repository
print("📦 Deploying Data Agent items...")

# Deploy Data Agent items using launcher.deploy_artifacts
deployer = launcher.deploy_artifacts(
    repository_directory=workspace_directory,
    item_types=["DataAgent"],
    allow_non_empty_workspace=True,
    deployment_retries=2
)

print("✅ Data Agent items deployed successfully")

In [ ]:
# Move current notebook to the notebooks folder
move_item_to_folder(
        item_name=notebookutils.runtime.context["currentNotebookName"],
        item_type="Notebook",
        folder_name="notebooks",
        workspace_id=workspace_id,
        client=client
    )
print("✅ Current notebook moved successfully!")

# Move all deployed ontologies to the target folder
print(f"\n📁 Moving ontologies to folder '{ONTOLOGY_TARGET_FOLDER}'...")
for ontology_name, ontology_id in deployed_ontology_ids:
    success = move_item_to_folder(
        item_name=ontology_name,
        item_type="Ontology",
        folder_name=ONTOLOGY_TARGET_FOLDER,
        workspace_id=workspace_id,
        client=client
    )
    if success:
        print(f"   ✅ Moved '{ontology_name}' to '{ONTOLOGY_TARGET_FOLDER}'")
    else:
        print(f"   ⚠️ Failed to move '{ontology_name}'")

print(f"\n✅ All ontologies moved to '{ONTOLOGY_TARGET_FOLDER}' folder")

## Next Steps
Your automated solution deployment is complete!

⚠️ Please be sure to refresh your browser window to reflect all newly-deployed items in the Fabric portal!

Review the [README](https://github.com/microsoft/microsoft-iq-solution-accelerator/?tab=readme-ov-file) document for details on running simulations and exploring your solution.